# Final Results Table and Paper-Ready Figures

**Author** : Anwesha Singh — CSE, Manipal University Jaipur

---

Run this notebook **last**, after all training and benchmark notebooks  
have been completed for every horizon.  

This notebook produces:
1. A unified cross-model, cross-horizon results table (CSV + LaTeX)  
2. Best-vs-second-best delta analysis  
3. A per-horizon improvement percentage over each baseline  
4. All publication-quality plots required for the paper


In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, PROJECT_ROOT)

from core.config import (
    FORECAST_HORIZONS, BENCHMARK_DIR, NIOA_RESULTS_DIR, RESULTS_DIR
)
from benchmarking.utils.metrics import compute_metrics

ANALYSIS_DIR = os.path.join(RESULTS_DIR, "analysis")
os.makedirs(ANALYSIS_DIR, exist_ok=True)

sns.set(style="whitegrid", palette="tab10", font_scale=1.1)
print("Final results notebook ready.")

---
## 1 · Load All Saved Metrics

In [ ]:
def load_nioa_metrics(nioa_root, horizon):
    """
    Find the most recent NiOA-DRNN experiment for the given horizon
    and load its metrics.json.
    """
    candidates = [
        d for d in os.listdir(nioa_root)
        if d.startswith(f"k{horizon}_")
    ]
    if not candidates:
        return None

    # Take the latest experiment (lexicographically largest timestamp)
    latest = sorted(candidates)[-1]
    metrics_path = os.path.join(nioa_root, latest, "metrics.json")
    if not os.path.isfile(metrics_path):
        return None

    with open(metrics_path) as f:
        m = json.load(f)
    return m


all_rows = []

for h in FORECAST_HORIZONS:
    horizon_dir = os.path.join(BENCHMARK_DIR, f"horizon_{h}")

    # ── Benchmark models ────────────────────────────────────────────────────
    if os.path.isdir(horizon_dir):
        for model_name in sorted(os.listdir(horizon_dir)):
            metrics_path = os.path.join(horizon_dir, model_name, "metrics.json")
            if os.path.isfile(metrics_path):
                with open(metrics_path) as f:
                    m = json.load(f)
                all_rows.append({"Model": model_name, "Horizon_s": h, **m})

    # ── Proposed NiOA-DRNN ──────────────────────────────────────────────────
    nioa_m = load_nioa_metrics(NIOA_RESULTS_DIR, h)
    if nioa_m:
        # Normalise key names (evaluate.py uses 'R2'; check for 'R²' too)
        row = {"Model": "nioa_drnn", "Horizon_s": h}
        row["MAE"]   = nioa_m.get("MAE",   nioa_m.get("mae",  np.nan))
        row["RMSE"]  = nioa_m.get("RMSE",  nioa_m.get("rmse", np.nan))
        row["R2"]    = nioa_m.get("R2",    nioa_m.get("r2",   np.nan))
        row["sMAPE"] = nioa_m.get("sMAPE", nioa_m.get("smape",np.nan))
        all_rows.append(row)

results = pd.DataFrame(all_rows)

# Model display-name mapping for the paper
MODEL_LABELS = {
    "linear_regression" : "Linear Regression",
    "svr"               : "SVR",
    "xgboost"           : "XGBoost",
    "mlp"               : "MLP",
    "arima"             : "ARIMA",
    "vanilla_lstm"      : "Vanilla LSTM",
    "cnn_lstm"          : "CNN-LSTM",
    "drnn_optuna"       : "DRNN + Optuna",
    "nioa_drnn"         : "DRNN + NiOA (Proposed)",
}
results["Model_Label"] = results["Model"].map(MODEL_LABELS).fillna(results["Model"])

print(f"Loaded results for {results['Model'].nunique()} models across {results['Horizon_s'].nunique()} horizons.")
print(results[["Model_Label", "Horizon_s", "MAE", "RMSE", "R2", "sMAPE"]].to_string(index=False))

---
## 2 · Paper Table — MAE (Pivot)

In [ ]:
# Model ordering for the paper (classical → statistical → DL → proposed)
MODEL_ORDER = [
    "Linear Regression", "SVR", "XGBoost", "MLP", "ARIMA",
    "Vanilla LSTM", "CNN-LSTM", "DRNN + Optuna", "DRNN + NiOA (Proposed)"
]

for metric in ["MAE", "RMSE", "R2", "sMAPE"]:
    pivot = (
        results
        .pivot(index="Model_Label", columns="Horizon_s", values=metric)
        .reindex([m for m in MODEL_ORDER if m in results["Model_Label"].values])
    )
    pivot.columns = [f"k={h}s" for h in pivot.columns]

    print(f"\n{'='*60}")
    print(f"{metric} Comparison")
    print('='*60)
    print(pivot.to_string(float_format="{:.4f}".format))

    # Save CSV
    pivot.to_csv(os.path.join(ANALYSIS_DIR, f"table_{metric.lower()}.csv"))

    # Save LaTeX
    latex_str = pivot.to_latex(
        float_format="{:.4f}".format,
        bold_rows=False,
        caption=f"{metric} comparison across models and forecast horizons.",
        label=f"tab:{metric.lower()}_comparison",
    )
    with open(os.path.join(ANALYSIS_DIR, f"table_{metric.lower()}.tex"), "w") as f:
        f.write(latex_str)

print(f"\nAll tables saved to : {ANALYSIS_DIR}")

---
## 3 · NiOA Improvement Over Each Baseline (%)

In [ ]:
nioa_rows = results[results["Model"] == "nioa_drnn"].set_index("Horizon_s")
improvement_rows = []

for _, row in results[results["Model"] != "nioa_drnn"].iterrows():
    h = row["Horizon_s"]
    if h not in nioa_rows.index:
        continue

    nioa_mae  = nioa_rows.loc[h, "MAE"]
    nioa_rmse = nioa_rows.loc[h, "RMSE"]
    base_mae  = row["MAE"]
    base_rmse = row["RMSE"]

    if base_mae > 0:
        pct_mae  = (base_mae  - nioa_mae)  / base_mae  * 100
        pct_rmse = (base_rmse - nioa_rmse) / base_rmse * 100
    else:
        pct_mae = pct_rmse = float("nan")

    improvement_rows.append({
        "Baseline" : row["Model_Label"],
        "Horizon"  : h,
        "MAE_improvement_%"  : round(pct_mae,  2),
        "RMSE_improvement_%" : round(pct_rmse, 2),
    })

imp_df = pd.DataFrame(improvement_rows)
if not imp_df.empty:
    imp_pivot = imp_df.pivot(index="Baseline", columns="Horizon", values="MAE_improvement_%")
    print("NiOA-DRNN MAE improvement (%) over each baseline:")
    print(imp_pivot.to_string(float_format="{:.2f}".format))
    imp_pivot.to_csv(os.path.join(ANALYSIS_DIR, "nioa_improvement_pct.csv"))
else:
    print("No comparison data available yet — run all benchmarks first.")

---
## 4 · Publication Figures

In [ ]:
# ── A · MAE vs Horizon (all models) ─────────────────────────────────────────
if not results.empty:
    fig, ax = plt.subplots(figsize=(11, 5))
    for label, grp in results.groupby("Model_Label"):
        ls = "-" if label == "DRNN + NiOA (Proposed)" else "--"
        lw = 2.5 if label == "DRNN + NiOA (Proposed)" else 1.5
        mk = "o"  if label == "DRNN + NiOA (Proposed)" else "s"
        ax.plot(grp["Horizon_s"].values, grp["MAE"].values,
                marker=mk, linewidth=lw, linestyle=ls,
                markersize=7, label=label)

    ax.set_xscale("log")
    ax.set_xticks(FORECAST_HORIZONS)
    ax.set_xticklabels([str(h) for h in FORECAST_HORIZONS])
    ax.set_xlabel("Forecast Horizon k (seconds)", fontsize=12)
    ax.set_ylabel("MAE (kWh)", fontsize=12)
    ax.set_title("MAE vs Forecast Horizon — All Models", fontsize=13)
    ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(ANALYSIS_DIR, "mae_vs_horizon_all.png"), dpi=200, bbox_inches="tight")
    plt.show(); plt.close()

# ── B · R² vs Horizon ───────────────────────────────────────────────────────
if not results.empty:
    fig, ax = plt.subplots(figsize=(11, 5))
    for label, grp in results.groupby("Model_Label"):
        ls = "-" if label == "DRNN + NiOA (Proposed)" else "--"
        lw = 2.5 if label == "DRNN + NiOA (Proposed)" else 1.5
        ax.plot(grp["Horizon_s"].values, grp["R2"].values,
                marker="o", linewidth=lw, linestyle=ls,
                markersize=7, label=label)

    ax.axhline(0, color="black", linestyle=":", linewidth=0.9)
    ax.set_xscale("log")
    ax.set_xticks(FORECAST_HORIZONS)
    ax.set_xticklabels([str(h) for h in FORECAST_HORIZONS])
    ax.set_xlabel("Forecast Horizon k (seconds)", fontsize=12)
    ax.set_ylabel("R²", fontsize=12)
    ax.set_title("R² vs Forecast Horizon — All Models", fontsize=13)
    ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(ANALYSIS_DIR, "r2_vs_horizon_all.png"), dpi=200, bbox_inches="tight")
    plt.show(); plt.close()

# ── C · Per-horizon grouped bar (MAE, for each available horizon) ────────────
available = sorted(results["Horizon_s"].unique())
if available:
    n_h = len(available)
    fig, axes = plt.subplots(1, n_h, figsize=(5 * n_h, 5), sharey=False)
    if n_h == 1:
        axes = [axes]

    for ax, h in zip(axes, available):
        sub = (
            results[results["Horizon_s"] == h]
            .sort_values("MAE")
            .reindex(
                [r for r in
                 results[results["Horizon_s"] == h]
                 .sort_values("MAE")["Model_Label"].tolist()]
            )
        )
        colours = [
            "darkorange" if m == "DRNN + NiOA (Proposed)" else "steelblue"
            for m in sub["Model_Label"]
        ]
        ax.barh(sub["Model_Label"], sub["MAE"], color=colours)
        ax.set_title(f"k = {h} s")
        ax.set_xlabel("MAE (kWh)")
        ax.invert_yaxis()

    plt.suptitle(
        "MAE Comparison — All Models (orange = proposed NiOA-DRNN)",
        fontsize=13, fontweight="bold", y=1.02
    )
    plt.tight_layout()
    plt.savefig(
        os.path.join(ANALYSIS_DIR, "mae_bar_per_horizon.png"),
        dpi=200, bbox_inches="tight"
    )
    plt.show(); plt.close()

print(f"All publication figures saved to : {ANALYSIS_DIR}")

---
## 5 · Final Printout

In [ ]:
print("\n" + "="*70)
print("  FINAL RESULTS SUMMARY")
print("="*70)

for h in sorted(results["Horizon_s"].unique()):
    print(f"\nHorizon k = {h} s")
    sub = (
        results[results["Horizon_s"] == h]
        [["Model_Label", "MAE", "RMSE", "R2", "sMAPE"]]
        .sort_values("MAE")
    )
    print(sub.to_string(index=False, float_format="{:.4f}".format))

print("\n" + "="*70)
print(f"Tables and figures : {ANALYSIS_DIR}")
for f in sorted(os.listdir(ANALYSIS_DIR)):
    print(f"  {f}")